# Simple workflow
Learn following using OpenAI framework
1. Agent workflow 
2. Tools and Handoffs


In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import asyncio
import os

load_dotenv(override=True)

True

# Step 1: Agent Workflow

In [2]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [3]:
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [4]:
# Test the connectivity to openai by running simple check

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC 2 Compliance with ComplAI

Dear [Recipient’s Name],

I hope this message finds you well. My name is [Your Name], and I'm with ComplAI, an AI-powered SaaS solution designed specifically to streamline SOC 2 compliance processes and enhance audit preparedness.

Navigating the complexities of SOC 2 compliance can be a daunting task, often diverting valuable resources away from your core business activities. ComplAI offers an intelligent platform that automates compliance workflows, provides real-time insights, and simplifies documentation, allowing you to focus on what you do best.

Here are a few ways our solution can benefit your organization:

- **Automated Compliance Management**: Reduce the manual workload with automated tracking and reporting.
- **Real-Time Insights**: Gain immediate visibility into your compliance posture.
- **Audit-Ready Documentation**: Elevate your readiness for audits with organized, easily accessible records.

I would love to schedule

### Run all 3 agents in parallel

In [5]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Streamline Your SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this message finds you well.

As organizations face increasingly complex requirements for data security and compliance, ensuring SOC 2 certification can be both time-consuming and resource-intensive. At ComplAI, we specialize in simplifying this process through our cutting-edge SaaS tool, designed specifically to streamline SOC 2 compliance and audit preparation.

Our AI-powered platform not only provides real-time insights but also automates documentation and workflows, reducing the burden on your team while enhancing accuracy. By integrating our solution, you can expect to:

- Accelerate your SOC 2 audit timelines
- Minimize manual errors
- Maintain continuous compliance with evolving standards

I would love the opportunity to discuss how ComplAI can support your compliance efforts and help you save valuable time and resources.

Are you available for a quick call next week? 

Thank you for consid

In [ ]:
# Agent to pick best email from the list

sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

## Generate 3 emails using 3 agents and pick the best one using another agent

In [7]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Let’s Make Your SOC2 Audit as Fun as a Root Canal! 🎉

Hi [Recipient's Name],

I hope this email finds you in good spirits, perhaps even sipping coffee while pretending to review compliance policies. 😊 

Let’s face it: SOC2 compliance can feel like trying to complete a puzzle with half the pieces missing while wearing a blindfold. Is that a crucial document or an ancient manuscript? Who knows! But fear not; I come bearing solutions from ComplAI—your trusty sidekick in the wild world of audits.

Our AI-powered SaaS tool doesn’t just untangle the compliance web; it practically dances through it! Forget about shuffling through endless spreadsheets and email chains. With ComplAI, you’ll be confidently strutting into your next audit, with everything prepared faster than you can say “Where’s my compliance handbook?!”

🪄 What we offer:
- Automated compliance checklists so you can relax (or finally tackle that binge-worthy series).
- Real-time status tracking to ensur

Now go and check out the trace:

https://platform.openai.com/traces

### Crete a tool using @function_tool decorator for the function

In [18]:
@function_tool
def send_email(body: str):
    """ This is a fake method and it just prints the email instead of sending it out.  In real world we could have send email routine using sendgrid or some other service/library """
    print("In send_email function" )
    return {"status": "success"}

In [20]:
# Lets look at the variable.. it should have created FunctionTool that can be sent to the model
send_email

FunctionTool(name='send_email', description='This is a fake method and it just prints the email instead of sending it out.  In real world we could have send email routine using sendgrid or some other service/library', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10ce6e980>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [21]:
instructions ="You are an email sender. You receive the body of an email to be sent. \
You use the send_email tool to send the email "

email_tools = [send_email]

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Send an email")


### Convert agents to tool.  Put all tools together into array


In [22]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent1.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent1.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3]

tools


[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10d862020>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10d8623e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='Write 

### Sales Manager agent is plannign agent.  It is given tools to generate 3 emails, instructions to select the best email and hand off agent to send the email

In [1]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Handoff for Sending: Pass ONLY the winning email draft to 'Email Manager' agent. The Email Manager will take care of sending email.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager", 
    instructions=instructions, 
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

NameError: name 'Agent' is not defined

### Now go and check out the trace to see all agents:

https://platform.openai.com/traces
